# 03 – Hypothesis Testing

> **Do AI Hype Cycles Create Temporary Stock Market Bubbles?**

This notebook implements all statistical hypothesis tests.
Hype is measured via a **combined index** of Google Trends + Reddit mentions.

### Hypotheses

| # | H0 | H1 |
|---|---|---|
| H1 | AI hype has no effect on stock returns | Hype periods → significantly higher returns |
| H2 | AI hype has no effect on volatility | Hype periods → significantly higher volatility |
| H3 | Hype(t) does not predict return(t+1) | Higher hype today → higher return tomorrow |
| H4 | Post-hype returns are not negative | Post-hype correction exists (bubble effect) |

### Tests Used
- **Welch two-sample t-test** — hype vs normal period comparisons
- **Pearson correlation** — hype vs next-day returns
- **Granger causality** — does hype Granger-cause returns?
- **Event study (CAR)** — cumulative abnormal returns around AI milestones
- **ADF test** — stationarity check

## CELL 1: Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import argrelextrema
from statsmodels.tsa.stattools import grangercausalitytests, adfuller

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in ["notebook", "notebooks"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES        = PROJECT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
TICKERS = ["NVDA", "MSFT", "AAPL"]
ALPHA   = 0.05
print(f"Setup complete. Significance level α = {ALPHA}")

## CELL 2: Load Data

In [ ]:
df     = pd.read_csv(DATA_PROCESSED / "merged_data.csv",  index_col="date", parse_dates=True)
events = pd.read_csv(DATA_PROCESSED / "ai_events.csv",    parse_dates=["date"])

print(f"Data loaded: {df.shape[0]} trading days")
print(f"Hype days  : {df['is_hype_period'].sum()}  |  Normal days: {(df['is_hype_period']==0).sum()}")
print(f"Hype sources: google_hype_index + reddit_hype_index → combined hype_index")
df[["hype_index", "google_hype_index", "reddit_hype_index",
    "NVDA_return", "MSFT_return", "AAPL_return"]].describe().round(5)

## CELL 3: H1 – Hype periods → higher stock returns?

**Test:** Welch two-sample t-test (one-tailed: hype mean > normal mean)

In [ ]:
print("═" * 60)
print("H1: AI hype periods → higher stock returns")
print("    Test: Welch two-sample t-test (one-tailed)")
print("═" * 60)

h1_rows = []
for ticker in TICKERS:
    col        = f"{ticker}_return"
    hype_ret   = df.loc[df["is_hype_period"] == 1, col].dropna()
    normal_ret = df.loc[df["is_hype_period"] == 0, col].dropna()

    t_stat, p_two = stats.ttest_ind(hype_ret, normal_ret, equal_var=False)
    p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    sig = "✓ REJECT H0" if p_one < ALPHA else "✗ FAIL TO REJECT"
    h1_rows.append({"ticker": ticker,
                    "hype_mean_%": hype_ret.mean()*100,
                    "normal_mean_%": normal_ret.mean()*100,
                    "diff_%": (hype_ret.mean()-normal_ret.mean())*100,
                    "t_stat": t_stat, "p_value": p_one, "result": sig})

    print(f"\n{ticker}:")
    print(f"  Hype mean   : {hype_ret.mean()*100:.4f}%/day")
    print(f"  Normal mean : {normal_ret.mean()*100:.4f}%/day")
    print(f"  t-stat      : {t_stat:.4f}")
    print(f"  p (one-tail): {p_one:.4f}  → {sig}")

print("\n── Summary ─────────────────────────────────────────────")
display(pd.DataFrame(h1_rows).set_index("ticker").round(6))

## CELL 4: H2 – Hype periods → higher volatility?

**Test:** Welch t-test on 20-day rolling volatility

In [ ]:
print("═" * 60)
print("H2: AI hype periods → higher stock volatility")
print("    Test: Welch t-test on annualised 20-day rolling volatility")
print("═" * 60)

h2_rows = []
for ticker in TICKERS:
    col      = f"{ticker}_volatility"
    hype_vol = df.loc[df["is_hype_period"] == 1, col].dropna()
    norm_vol = df.loc[df["is_hype_period"] == 0, col].dropna()

    t_stat, p_two = stats.ttest_ind(hype_vol, norm_vol, equal_var=False)
    p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    lev_stat, lev_p = stats.levene(hype_vol, norm_vol)

    sig = "✓ REJECT H0" if p_one < ALPHA else "✗ FAIL TO REJECT"
    h2_rows.append({"ticker": ticker,
                    "hype_vol": hype_vol.mean(), "normal_vol": norm_vol.mean(),
                    "t_stat": t_stat, "p_value": p_one,
                    "levene_p": lev_p, "result": sig})

    print(f"\n{ticker}:")
    print(f"  Hype volatility  : {hype_vol.mean():.4f}")
    print(f"  Normal volatility: {norm_vol.mean():.4f}")
    print(f"  t-stat           : {t_stat:.4f}")
    print(f"  p (one-tail)     : {p_one:.4f}  → {sig}")
    print(f"  Levene p         : {lev_p:.4f}")

print("\n── Summary ─────────────────────────────────────────────")
display(pd.DataFrame(h2_rows).set_index("ticker").round(6))

## CELL 5: H3 – Does today's hype predict tomorrow's return?

**Test 1:** Pearson correlation — `hype_index(t)` vs `return(t+1)`  
**Test 2:** Granger causality (max lag = 5 days)

In [ ]:
print("═" * 60)
print("H3: Today's hype predicts tomorrow's return")
print("    Test: Pearson correlation (hype lag-1 vs next-day return)")
print("═" * 60)

h3_rows = []
for ticker in TICKERS:
    hype_lag1   = df["hype_index"].shift(1)
    next_return = df[f"{ticker}_return"]
    valid = pd.concat([hype_lag1, next_return], axis=1).dropna()
    r, p  = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])

    sig = "✓ REJECT H0" if p < ALPHA else "✗ FAIL TO REJECT"
    h3_rows.append({"ticker": ticker, "pearson_r": r, "p_value": p, "result": sig})

    print(f"\n{ticker}:")
    print(f"  Pearson r : {r:.4f}")
    print(f"  p-value   : {p:.4f}  → {sig}")

print("\n── Summary ─────────────────────────────────────────────")
display(pd.DataFrame(h3_rows).set_index("ticker").round(6))

In [ ]:
print("═" * 60)
print("H3 (cont.): Granger Causality Test — hype → stock returns")
print("    Max lag: 5 trading days")
print("═" * 60)

MAX_LAG = 5
for ticker in TICKERS:
    print(f"\n── {ticker} ──")
    granger_df = df[[f"{ticker}_return", "hype_index"]].dropna()
    try:
        gc = grangercausalitytests(granger_df, maxlag=MAX_LAG, verbose=False)
        for lag in range(1, MAX_LAG + 1):
            p_f = gc[lag][0]["ssr_ftest"][1]
            sig = "✓" if p_f < ALPHA else "✗"
            print(f"  Lag {lag}: p = {p_f:.4f}  {sig}")
    except Exception as e:
        print(f"  Error: {e}")

## CELL 6: H4 – Event Study: Cumulative Abnormal Returns (CAR)

Abnormal return = actual return − expected return (60-day rolling baseline).  
We examine the window [−10, +30] days around each major AI event.

In [ ]:
# Compute abnormal returns
WINDOW = 60
for ticker in TICKERS:
    col = f"{ticker}_return"
    df[f"{ticker}_expected"] = df[col].rolling(WINDOW, min_periods=20).mean().shift(1)
    df[f"{ticker}_abnormal"] = df[col] - df[f"{ticker}_expected"]

PRE_DAYS  = 10
POST_DAYS = 30

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(12, 4*len(TICKERS)), sharex=False)
fig.suptitle("Cumulative Abnormal Returns Around Major AI Events", fontsize=13, fontweight="bold")

COLORS = {"NVDA": "#76b900", "MSFT": "#00a4ef", "AAPL": "#555555", "hype": "#e74c3c"}

for ax, ticker in zip(axes, TICKERS):
    col = f"{ticker}_abnormal"
    for _, row in events.iterrows():
        event_date = row["date"]
        if event_date < df.index.min() or event_date > df.index.max():
            continue
        idx   = df.index.searchsorted(event_date)
        start = max(0, idx - PRE_DAYS)
        end   = min(len(df), idx + POST_DAYS + 1)
        window_ret   = df[col].iloc[start:end].values
        event_offset = idx - start
        car  = np.cumsum(window_ret)
        if event_offset < len(car):
            car = car - car[event_offset]
        days = np.arange(-event_offset, len(car) - event_offset)
        ax.plot(days, car * 100, alpha=0.6, linewidth=1.2, label=row["event"])

    ax.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Event day")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{ticker}", fontsize=11)
    ax.set_xlabel("Days relative to event")
    ax.set_ylabel("CAR (%)")
    ax.legend(fontsize=7, loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES / "09_event_study_CAR.png", bbox_inches="tight")
plt.show()

## CELL 7: H4 – Post-Hype Price Correction Test

**Test:** One-sample t-test — are post-hype-peak returns significantly negative?

In [ ]:
print("═" * 60)
print("H4: Post-hype price correction (bubble effect)")
print("    Test: One-sample t-test (mean return < 0 after hype peaks)")
print("═" * 60)

# Detect local hype peaks (at least 20-day separation)
hype_vals  = df["hype_index"].values
peak_idxs  = argrelextrema(hype_vals, np.greater, order=20)[0]
peak_dates = df.index[peak_idxs]
print(f"\nDetected {len(peak_dates)} hype peaks.")

POST_WINDOW = 30  # days after each peak

for ticker in TICKERS:
    col = f"{ticker}_return"
    post_returns = []
    for peak_date in peak_dates:
        idx = df.index.searchsorted(peak_date)
        end = min(len(df), idx + POST_WINDOW + 1)
        post_returns.extend(df[col].iloc[idx+1:end].dropna().values)

    post_returns = np.array(post_returns)
    t_stat, p_two = stats.ttest_1samp(post_returns, popmean=0)
    p_neg = p_two / 2 if t_stat < 0 else 1 - p_two / 2

    sig = "✓ REJECT H0 (correction exists)" if p_neg < ALPHA else "✗ FAIL TO REJECT"
    print(f"\n{ticker} – {len(post_returns)} post-peak observations:")
    print(f"  Mean return : {post_returns.mean()*100:.4f}%/day")
    print(f"  t-statistic : {t_stat:.4f}")
    print(f"  p (mean<0)  : {p_neg:.4f}  → {sig}")

## CELL 8: Stationarity Check – ADF Test

In [ ]:
print("ADF Stationarity Test (H0: unit root = non-stationary)")
print("─" * 60)

test_cols = ["NVDA_return", "MSFT_return", "AAPL_return",
             "hype_index", "google_hype_index", "reddit_hype_index"]

for col in test_cols:
    series = df[col].dropna()
    adf_stat, p_val, _, _, crit, _ = adfuller(series, autolag="AIC")
    result = "✓ Stationary" if p_val < ALPHA else "✗ Non-stationary"
    print(f"{col:<28} ADF={adf_stat:7.3f}  p={p_val:.4f}  → {result}")

## CELL 9: Full Hypothesis Testing Summary

In [ ]:
summary = pd.DataFrame([
    {"#": "H1", "Question": "Hype periods → higher returns?",
     "Test": "Welch t-test", "See cell": 3},
    {"#": "H2", "Question": "Hype periods → higher volatility?",
     "Test": "Welch t-test + Levene", "See cell": 4},
    {"#": "H3", "Question": "Hype(t) predicts return(t+1)?",
     "Test": "Pearson r + Granger causality", "See cell": "5–6"},
    {"#": "H4", "Question": "Post-hype correction exists?",
     "Test": "Event study (CAR) + one-sample t-test", "See cell": "7"},
])

print("\n══════════════════════════════════════════════════════")
print("  HYPOTHESIS TESTING SUMMARY")
print("══════════════════════════════════════════════════════")
display(summary.set_index("#"))
print(f"\nAll tests use α = {ALPHA}")
print("Hype measured by: Google Trends + Reddit mentions (combined index)")